In [ ]:
!pip install kagglehub tensorflow opencv-python matplotlib numpy pandas

In [ ]:
import kagglehub
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import cv2
import os
import random
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models

In [ ]:
path = kagglehub.dataset_download("iarunava/cell-images-for-detecting-malaria")
print(path)
DATASET_PATH = os.path.join(path, "cell_images")
print(os.listdir(DATASET_PATH))

In [ ]:
IMG_SIZE = 64
BATCH = 128
EPOCHS = 2

In [ ]:
datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train = datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH,
    class_mode="binary",
    classes=['Parasitized', 'Uninfected'],
    subset="training"
)

val = datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH,
    class_mode="binary",
    classes=['Parasitized', 'Uninfected'],
    subset="validation"
)

In [ ]:
inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

x = layers.Conv2D(32, (3,3), activation='relu')(inputs)
x = layers.MaxPooling2D()(x)

x = layers.Conv2D(64, (3,3), activation='relu')(x)
x = layers.MaxPooling2D()(x)

x = layers.Conv2D(128, (3,3), activation='relu', name="last_conv")(x)
x = layers.MaxPooling2D()(x)

x = layers.Flatten()(x)
x = layers.Dense(128, activation='relu')(x)
outputs = layers.Dense(1, activation='sigmoid')(x)

model = models.Model(inputs, outputs)

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
history = model.fit(
    train,
    validation_data=val,
    epochs=EPOCHS
)

In [ ]:
plt.plot(history.history['accuracy'], marker='o')
plt.plot(history.history['val_accuracy'], marker='o')
plt.legend(["Train","Validation"])
plt.title("Model Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.show()

In [ ]:
test_paths = []

for cls in ['Parasitized', 'Uninfected']:
    cls_path = os.path.join(DATASET_PATH, cls)
    if os.path.isdir(cls_path):
        imgs = os.listdir(cls_path)
        for i in random.sample(imgs, 2):
            test_paths.append(os.path.join(cls_path, i))

print(test_paths)

In [ ]:
def gradcam(img_path, model):
    img = tf.keras.preprocessing.image.load_img(
        img_path, target_size=(IMG_SIZE, IMG_SIZE)
    )
    img_array = tf.keras.preprocessing.image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    grad_model = tf.keras.models.Model(
        model.input,
        [model.get_layer("last_conv").output, model.output]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        loss = predictions[:, 0]

    grads = tape.gradient(loss, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0,1,2))

    conv_outputs = conv_outputs[0]
    heatmap = tf.reduce_sum(conv_outputs * pooled_grads, axis=-1)

    heatmap = tf.maximum(heatmap, 0)
    heatmap /= tf.reduce_max(heatmap) + 1e-8

    heatmap = heatmap.numpy()
    heatmap = cv2.resize(heatmap, (IMG_SIZE, IMG_SIZE))

    return img, heatmap

In [ ]:
for path in test_paths:
    img, heatmap = gradcam(path, model)
    img = np.array(img)

    heatmap = np.uint8(255 * heatmap)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)

    superimposed = heatmap * 0.4 + img

    plt.figure(figsize=(10,3))

    plt.subplot(1,3,1)
    plt.imshow(img)
    plt.axis("off")

    plt.subplot(1,3,2)
    plt.imshow(heatmap)
    plt.axis("off")

    plt.subplot(1,3,3)
    plt.imshow(superimposed.astype("uint8"))
    plt.axis("off")

    plt.show()